# Module 2 · Demo — Your First Agent

**From 0 to Agentic AI — DataHack Summit 2026**

In the previous demo the model could only *request* a tool. Here we hand a prebuilt
agent a **real** tool — **web search** — and watch it answer a question end-to-end,
breaking the *can't act* wall from the last notebook.

> This is a **teaching demo**, not the project. We build the real Knowledge Assistant
> properly with LangGraph starting in **Module 4** — this is just to *see* an agent work.

### What it does
- Takes a question
- Decides whether it needs to **search the web**
- Calls the search tool, reads the results, and answers — **grounded**, with a trace we can inspect

---
## Setup

In [1]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" \
               "langgraph>=1.0,<2" "langchain-tavily>=0.2"

ERROR: Could not find a version that satisfies the requirement langchain<2,>=1.2 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.0.20, 0.0.21, 0.0.22, 0.0.23, 0.0.24, 0.0.25, 0.0.26, 0.0.27, 0.0.28, 0.0.29, 0.0.30, 0.0.31, 0.0.32, 0.0.33, 0.0.34, 0.0.35, 0.0.36, 0.0.37, 0.0.38, 0.0.39, 0.0.40, 0.0.41, 0.0.42, 0.0.43, 0.0.44, 0.0.45, 0.0.46, 0.0.47, 0.0.48, 0.0.49, 0.0.50, 0.0.51, 0.0.52, 0.0.53, 0.0.54, 0.0.55, 0.0.56, 0.0.57, 0.0.58, 0.0.59, 0.0.60, 0.0.61, 0.0.63, 0.0.64, 0.0.65, 0.0.66, 0.0.67, 0.0.68, 0.0.69, 0.0.70, 0.0.71, 0.0.72, 0.0.73, 0.0.74, 0.0.75, 0.0.76, 0.0.77, 0.0.78, 0.0.79, 0.0.80, 0.0.81, 0.0.82, 0.0.83, 0.0.84, 0.0.85, 0.0.86, 0.0.87, 0.0.88, 0.0.89, 0.0.90, 0.0.91, 0.0.92, 0.0.93, 0.0.94, 0.0.95, 0.0.96, 0.0.97, 0.0.98, 0.0.99rc0, 0.0.99, 0.0.100, 0.0.101rc0, 0.0.101, 0.0.102rc0, 0.0.102, 0.0.103, 0.0.104, 0.0.105, 0.0.106, 0.0.107, 0.0.108, 0.0.109, 0.0

In [2]:
import os
from getpass import getpass

# Local: load keys from src/.env. Colab: you'll be prompted for missing keys.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

# This demo needs an LLM key and a web-search key.
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · The web-search tool

`TavilySearch` is a ready-made LangChain tool built for LLMs — it returns clean search results the agent can read.

In [3]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=3)

# Quick sanity check — call the tool directly:
hits = web_search.invoke({"query": "What is LangGraph?"})
print(type(hits))
print(str(hits)[:400], "...")

<class 'dict'>
{'query': 'What is LangGraph?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.geeksforgeeks.org/machine-learning/what-is-langgraph', 'title': 'What is LangGraph - GeeksforGeeks', 'content': 'LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define w ...


---
## Step 2 · Assemble the agent

A system prompt gives the agent its role; `create_agent` runs the loop.

In [4]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)

SYSTEM_PROMPT = (
    "You are a helpful assistant for a software company. "
    "Answer questions about tools, services, and operations. "
    "Use web search when you need current or external information, "
    "and cite what you found. If unsure, say so."
)

agent = create_agent(
    llm,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
)

---
## Step 3 · Ask it something

A little helper to run a question and print the final answer.

In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [("user", question)]})
    return result["messages"][-1].content, result

answer, result = ask("Our team is evaluating LangGraph. What is it, and what is its latest major version?")
print(answer)

LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define workflows as nodes and edges, making complex agent interactions more structured, scalable, and easier to control. LangGraph represents an agent workflow as a stateful directed graph where nodes are Python functions, model calls, or tool invocations, and edges define the control flow between them. It is used for building stateful, multi-step AI agent workflows and is suitable for complex applications like customer service workflows.

The latest major version of LangGraph is 1.0, which marks the first stable major release of this durable agent framework, making it production-ready. This release comes after more than a year of iteration and adoption by companies like Uber, LinkedIn, and Klarna.


### Inspect the trace
See the agent decide to search, read the results, then answer — the reason → act → observe loop.

In [6]:
for m in result['messages']:
    m.pretty_print()

================================ Human Message =================================

Our team is evaluating LangGraph. What is it, and what is its latest major version?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_TfdDvcs9KrFhfVlzkRSLEKL8)
 Call ID: call_TfdDvcs9KrFhfVlzkRSLEKL8
  Args:
    query: What is LangGraph?
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "What is LangGraph?", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.geeksforgeeks.org/machine-learning/what-is-langgraph", "title": "What is LangGraph - GeeksforGeeks", "content": "LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define workflows as nodes and edges, making complex agent interactions more structured, scalable 

### Try your own
Swap in any question. Notice it only searches when it actually needs to.

In [7]:
answer, _ = ask("What does the term \"retrieval-augmented generation\" mean?")
print(answer)

Retrieval-augmented generation (RAG) is a technique in natural language processing that combines retrieval-based methods with generative models. In this approach, a system first retrieves relevant information or documents from a large external knowledge base or dataset based on a user's query. Then, it uses a generative model, such as a transformer-based language model, to produce a coherent and contextually appropriate response by incorporating the retrieved information.

The key idea is to enhance the generation process with factual and up-to-date information that the generative model alone might not have been trained on or might have forgotten. This helps improve the accuracy, relevance, and informativeness of the generated text, especially for tasks like question answering, summarization, and dialogue systems.

In summary, retrieval-augmented generation leverages both retrieval of external knowledge and the creative capabilities of generative models to produce better and more infor

---
## What this agent can — and can't — do yet

✅ It **acts**: chooses and runs a tool, then answers from real results — the *can't act* wall, gone.

🚧 But it's still a black box, and two walls remain:
- ❌ We didn't **build the loop** — it's prebuilt. Module 3 opens it up by hand.
- ❌ No **internal knowledge** or **memory** yet — those come with the real build (Modules 4+).

Next we stop pressing the easy button and **understand** the loop it runs — the **ReAct pattern**.

---
## Key takeaways
- A prebuilt agent + one real tool (web search) already answers questions end-to-end.
- The message trace *is* the loop: request → run → observe → answer.
- This was a demo to *see* an agent work — the real Knowledge Assistant gets built on LangGraph from Module 4.